In [1]:
import os
import torch
import pandas as pd
from PIL import Image
from pathlib import Path
from tqdm import tqdm
from transformers import BlipProcessor, BlipForConditionalGeneration

def generate_cloth_captions(csv_path, cloth_dir, output_dir, is_test_pairs=False):
    """
    Sinh caption tự động cho ảnh áo bằng mô hình BLIP. 
    Chỉ chạy trên danh sách ID đã được lọc sạch từ EDA.
    """
    # 1. Khởi tạo đường dẫn và tạo thư mục đầu ra nếu chưa có
    cloth_dir = Path(cloth_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # 2. Đọc danh sách ID sạch
    df = pd.read_csv(csv_path)
    if is_test_pairs and 'cloth_id' in df.columns:
        # Nếu là file test, ta cần lấy cột cloth_id
        ids = df['cloth_id'].unique().tolist() 
    else:
        # Nếu là file train, ta lấy cột id
        ids = df['id'].tolist()

    print(f"[*] Đã tải {len(ids)} ID hợp lệ từ file CSV.")

    # 3. Ưu tiên dùng GPU để chạy nhanh hơn
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"[*] Đang sử dụng thiết bị: {device.upper()}")

    # 4. Tải mô hình BLIP (Base)
    print("Đang tải mô hình Salesforce/blip-image-captioning-base...")
    processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
    model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")
    model.to(device)
    model.eval() # Chuyển sang chế độ suy luận (không train BLIP)

    # 5. Vòng lặp sinh Caption
    print("[*] Bắt đầu sinh Caption...")
    success_count = 0
    error_count = 0

    # Dùng tqdm để hiển thị thanh tiến trình
    pbar = tqdm(ids, desc="Generating Captions")
    for s_id in pbar:
        base_id = Path(s_id).stem
        img_path = cloth_dir / f"{base_id}.jpg"
        txt_path = output_dir / f"{base_id}.txt"
        
        # Bỏ qua nếu file text đã tồn tại 
        if txt_path.exists():
            success_count += 1
            continue
            
        if not img_path.exists():
            error_count += 1
            continue

        try:
            # Mở ảnh và chuẩn bị dữ liệu
            raw_image = Image.open(img_path).convert('RGB')
            
            # Khuyến nghị của BLIP: Thêm đoạn text mồi (conditional prompt)
            # Giúp mô hình tập trung vào việc mô tả cái áo thay vì phông nền
            text_prompt = "a professional studio photo of" 
            
            # Đưa qua processor và đẩy lên GPU
            inputs = processor(raw_image, text_prompt, return_tensors="pt").to(device)
            
            # Sinh văn bản
            with torch.no_grad(): # Tắt tính gradient để tiết kiệm VRAM
                out = model.generate(**inputs, max_new_tokens=40,
                                     min_length=5,                   # Đảm bảo không quá ngắn
                                     do_sample=True,                 # Bật lấy mẫu để tăng sự đa dạng
                                     top_k=50,                       # Lọc top từ phổ biến
                                     top_p=0.9,                      # Nucleus sampling
                                     repetition_penalty=1.2, # Ngăn chặn việc lặp lại một từ quá nhiều lần
                                     no_repeat_ngram_size=3  # Không cho phép lặp lại cụm 3 từ giống hệt nhau
                                    )
                
            # Giải mã Tensor thành String
            caption = processor.decode(out[0], skip_special_tokens=True)
            
            # Làm sạch chuỗi (Bỏ khoảng trắng thừa)
            caption = caption.strip().lower()

            # Lưu vào file .txt (CHÚ Ý: lưu raw_caption, việc thêm trigger_word đã có Class Preprocess lo)
            with open(txt_path, 'w', encoding='utf-8') as f:
                f.write(caption)
                
            success_count += 1
            # Tính toán thông số VRAM
            used_vram = torch.cuda.memory_allocated(device) / 1024**3
            reserved_vram = torch.cuda.memory_reserved(device) / 1024**3
            
            # Cập nhật thông số hiện ra bên cạnh thanh tiến trình tqdm
            pbar.set_postfix({
                "VRAM_Used": f"{used_vram:.2f}GB",
                "Reserved": f"{reserved_vram:.2f}GB"
            })
        except Exception as e:
            print(f"\n[Lỗi] ID {base_id}: {str(e)}")
            error_count += 1

    print("\n" + "="*40)
    print(f" HOÀN THÀNH SINH CAPTION")
    print(f" Thành công: {success_count}/{len(ids)}")
    print(f" Lỗi/Không tìm thấy ảnh: {error_count}")
    print(f" Output lưu tại: {output_dir}")
    print("="*40)


if __name__ == "__main__":
    TRAIN_CSV_PATH = "/kaggle/input/datasets/cthnhoddt/dlp-cleandatacsv/clean_vto_dataset_train.csv"
    # Tạo caption cho ảnh train
    CLOTH_DIR_TRAIN = "/kaggle/input/datasets/marquis03/high-resolution-viton-zalando-dataset/train/cloth"
    OUTPUT_DIR_TRAIN = "/kaggle/working/cloth-captions/train"
    generate_cloth_captions(TRAIN_CSV_PATH, CLOTH_DIR_TRAIN, OUTPUT_DIR_TRAIN)

    print("-------------------------------------------------------------------------------------\n")

    # Tạo caption cho ảnh test
    TEST_CSV_PATH = "/kaggle/input/datasets/cthnhoddt/dlp-cleandatacsv/clean_vto_dataset_test.csv"
    CLOTH_DIR_TEST = "/kaggle/input/datasets/marquis03/high-resolution-viton-zalando-dataset/test/cloth"
    OUTPUT_DIR_TEST = "/kaggle/working/cloth-captions/test"
    generate_cloth_captions(TEST_CSV_PATH, CLOTH_DIR_TEST, OUTPUT_DIR_TEST, is_test_pairs=True)


[*] Đã tải 9520 ID hợp lệ từ file CSV.
[*] Đang sử dụng thiết bị: CUDA
Đang tải mô hình Salesforce/blip-image-captioning-base...


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

[*] Bắt đầu sinh Caption...


Generating Captions: 100%|██████████| 1661/1661 [06:48<00:00,  4.06it/s, VRAM_Used=0.94GB, Reserved=1.07GB]


 HOÀN THÀNH SINH CAPTION
 Thành công: 1661/1661
 Lỗi/Không tìm thấy ảnh: 0
 Output lưu tại: /kaggle/working/cloth-captions/test


Dùng tên ảnh áo để đặt cho caption